In [26]:
!pip install -q mistralai==2.1.3 langchain==1.2.13 langchain-core==1.2.22 langchain-mistralai==1.1.2 python-dotenv==1.2.2 pydantic==2.12.5

In [27]:
from enum import Enum
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from mistralai.client import Mistral
from google.colab import userdata

In [28]:
# какой-то бесплатный клиент. подойдет любой
mistral_key = userdata.get('MISTRAL_API_KEY')

mistral_client = Mistral(api_key=mistral_key) if mistral_key else None

for name, client in [("Mistral", mistral_client)]:
    print(f"{'✅' if client else '❌'} {name} client {'ready' if client else 'not available'}")


✅ Mistral client ready


In [29]:
def _build_messages(user_prompt: str, system_prompt=None) -> list:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    return messages


def call_mistral(model_name: str, user_prompt: str, system_prompt=None,
                 temperature=None, top_p=None, max_tokens=None, **kwargs) -> str:
    messages = _build_messages(user_prompt, system_prompt)
    params = {k: v for k, v in dict(temperature=temperature, top_p=top_p, max_tokens=max_tokens).items() if v is not None}
    response = mistral_client.chat.complete(model=model_name, messages=messages, **params, **kwargs)
    return response.choices[0].message.content

def parse_mistral(model_name: str, user_prompt: str, response_format, system_prompt=None):
    messages = _build_messages(user_prompt, system_prompt)
    response = mistral_client.chat.parse(model=model_name, messages=messages, response_format=response_format)
    return response.choices[0].message.parsed

In [30]:
# Utility: Pretty-print any Pydantic object as a nested tree

def pretty_print(obj, indent=0, name=None):
    """Print a Pydantic object (or dict/list) as a nicely indented tree."""
    prefix = "  " * indent
    connector = "├── " if indent > 0 else ""

    if hasattr(obj, 'model_dump'):
        label = name or obj.__class__.__name__
        print(f"{prefix}{connector}{label}:")
        fields = obj.model_fields
        items = list(fields.keys())
        for i, field_name in enumerate(items):
            value = getattr(obj, field_name)
            is_last = (i == len(items) - 1)
            pretty_print(value, indent + 1, name=field_name)
    elif isinstance(obj, list):
        label = name or "list"
        if len(obj) == 0:
            print(f"{prefix}{connector}{label}: []")
        elif hasattr(obj[0], 'model_dump'):
            print(f"{prefix}{connector}{label}: ({len(obj)} items)")
            for i, item in enumerate(obj):
                pretty_print(item, indent + 1, name=f"[{i}]")
        else:
            print(f"{prefix}{connector}{label}: {obj}")
    else:
        label = name or "value"
        print(f"{prefix}{connector}{label}: {obj}")

═══════════════════════════════
# USER PROMPT
═══════════════════════════════

In [31]:
user_prompt = """

"""


═══════════════════════════════
# PROMPT ELEMENTS
═══════════════════════════════

═══════════════════════════════
# 1.COMMON TASK
═══════════════════════════════

In [32]:
common_task = """

═══════════════════════════════
COMMON TASK
═══════════════════════════════
You are an expert job-search query parser. Your task is to extract structured
information from a user's free-text message or uploaded resume into the
JobAndCandidateDescription schema.

Return ONLY a valid JSON object. No extra text, no markdown, no explanation.

Always return a single JSON object matching JobAndCandidateDescription
"""

═══════════════════════════════
# 2.GENERAL RULES
═══════════════════════════════

In [33]:
general_rules = """

═══════════════════════════════
GENERAL RULES
═══════════════════════════════
- Default every Optional field to null and every List field to [] unless the
  value is explicitly stated in the input.
- Do NOT infer, guess, or derive values that are not explicitly stated.
- Do NOT carry over past-tense experience into future-looking preferences and
  vice versa.
- When a value is ambiguous between candidate fact and job preference, apply
  the rule stated in the field's algorithm below.
- Preserve the user's original wording unless a normalization rule applies.

Two top-level models split the output:
  candidate_description — factual, stable information ABOUT the candidate
  job_description       — what the candidate IS LOOKING FOR in a new role

"""

═══════════════════════════════
# 3.FIELD-SPECIFIC ALGORITHMS
═══════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.1. CandidateDescription
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [34]:
candidate_description = f"""

═══════════════════════════════
FIELD-SPECIFIC ALGORITHMS
═══════════════════════════════
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL: CandidateDescription
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

----- candidate_description.years_experience -----

Extract the total number of years of professional work experience explicitly
stated by the user. Do NOT compute or estimate it from dates.

Valid triggers:
  "5 years of experience"
  "over 10 years in software development"
  "3+ years working in data"

Invalid triggers:
  Implied duration from resume dates (do NOT compute 2019–2024 = 5)
  Education duration
  Project duration

Algorithm:
  Step 1: Find an explicit numeric mention of years of experience.
  Step 2: Round to the nearest integer ("3+" → 3, "over 10" → 10).
  Step 3: If no explicit mention → null.

──────────────────────
CoT example
──────────────────────
Input: "I've been working as a backend engineer since 2018, mainly in Python."
  Detect explicit mention → NONE (only a start year is given, not a stated count)
  Do NOT compute 2026 − 2018 = 8
  → years_experience = null

──────────────────────
Few-shots
──────────────────────
Input: "7 years of experience in machine learning."
Output: {{ "years_experience": 7 }}

Input: "Senior engineer with 10+ years in the industry."
Output: {{ "years_experience": 10 }}

Input: "I started my career at Google in 2015."
Output: {{ "years_experience": null }}


----- candidate_description.languages -----

List natural languages the candidate explicitly states they can speak, read, or
work in. Do NOT include programming languages here.

Valid triggers:
  "fluent in English and German"
  "native Russian speaker"
  "working proficiency in French"

Invalid triggers:
  Programming / technical languages (Python, SQL → go to skills)
  Desired job languages ("I want a job in English" → not a candidate fact)

Algorithm:
  Step 1: Find all natural language mentions tied to the candidate.
  Step 2: Exclude programming languages.
  Step 3: Normalize to the language name in English (e.g. "немецкий" → "German").
  Step 4: Deduplicate, preserve order.
  Step 5: If none → [].

──────────────────────
Few-shots
──────────────────────
Input: "I speak Russian and English. I also know some Python and SQL."
Output: {{ "languages": ["Russian", "English"] }}

Input: "Native Spanish speaker, B2 level in English."
Output: {{ "languages": ["Spanish", "English"] }}

Input: "Looking for a job in an English-speaking company."
Output: {{ "languages": [] }}


----- candidate_description.education_level -----

The highest completed education level explicitly stated. Return a normalized
string: "high school", "bachelor's", "master's", "PhD", "vocational", or the
user's own phrase if it doesn't fit the above.

Valid triggers:
  "MSc in Computer Science"  → "master's"
  "finished my bachelor's"   → "bachelor's"
  "PhD in linguistics"       → "PhD"

Invalid triggers:
  Ongoing / in-progress education ("currently studying for master's" → null)
  Certifications and courses (→ skills, not education_level)

Algorithm:
  Step 1: Detect education mentions.
  Step 2: Keep only completed degrees.
  Step 3: Map to the normalized value.
  Step 4: If multiple completed degrees → keep the highest.
  Step 5: If none explicitly completed → null.

──────────────────────
Few-shots
──────────────────────
Input: "I have a bachelor's in economics and am currently finishing my master's."
Output: {{ "education_level": "bachelor's" }}

Input: "PhD graduate from MIT."
Output: {{ "education_level": "PhD" }}

Input: "Took several Coursera courses in ML."
Output: {{ "education_level": null }}


----- candidate_description.skills -----

List all technical and soft skills explicitly mentioned by the candidate as
things they know or have experience with. Keep the user's exact wording.

Valid triggers:
  "I know Python, SQL, and Docker"
  "experienced with React and TypeScript"
  "strong communication and leadership skills"

Invalid triggers:
  Desired skills ("I want to learn Rust" → not a current skill)
  Vague adjectives without a named skill ("hard-working", "motivated")

Algorithm:
  Step 1: Extract all named skills.
  Step 2: Exclude forward-looking ("want to learn") mentions.
  Step 3: Deduplicate, preserve order.
  Step 4: If none → [].

──────────────────────
Few-shots
──────────────────────
Input: "Proficient in Python and SQL. I want to learn Rust."
Output: {{ "skills": ["Python", "SQL"] }}

Input: "I have experience with Kubernetes, Terraform, and AWS."
Output: {{ "skills": ["Kubernetes", "Terraform", "AWS"] }}

"""

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.2. JobDescription
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [35]:
job_description = """

═══════════════════════════════
FIELD-SPECIFIC ALGORITHMS
═══════════════════════════════
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL: JobDescription
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

----- job_description.desired_positions -----

List of job titles, role names, or position types the candidate explicitly
states they are looking for. Keep the user's exact wording.

Valid triggers:
  "I want to find a job as a data scientist"   → ["data scientist"]
  "looking for backend or fullstack roles"     → ["backend engineer", "fullstack engineer"]
  "open to PM or TPM positions"               → ["PM", "TPM"]

Invalid triggers:
  Current or past titles ("I am currently a data analyst")
  Vague intents without a named role ("I want a new job")

Algorithm:
  Step 1: Detect future-oriented role mentions.
  Step 2: Exclude current/past titles.
  Step 3: Preserve the user's wording; split compound mentions into separate items.
  Step 4: Deduplicate.
  Step 5: If none → [].

Note: default_factory=list is used; the field is effectively [] when absent,
not null. Always return a list.

──────────────────────
CoT example
──────────────────────
Input: "I am a data analyst but I want to become a data scientist or ML engineer."
  "data analyst" → current title → exclude
  "data scientist" → desired → include
  "ML engineer" → desired → include
  → desired_positions = ["data scientist", "ML engineer"]

──────────────────────
Few-shots
──────────────────────
Input: "I want to find a job as a data scientist."
Output: {{ "desired_positions": ["data scientist"] }}

Input: "Searching for backend engineer or DevOps roles."
Output: {{ "desired_positions": ["backend engineer", "DevOps"] }}

Input: "Ready for a change, currently a QA engineer."
Output: {{ "desired_positions": [] }}

----- job_description.desired_tech_stack -----

List of technologies, tools, frameworks, or technical environments the
candidate explicitly states they WANT TO WORK WITH in the new role.
This captures future preference, not current skill.

Distinction from candidate_description.skills:
  skills             — what the candidate already knows (past/present fact)
  desired_tech_stack — what the candidate wants to use at the new job (future preference)

When both signals are present for the same technology, add it to BOTH fields.

Valid triggers:
  "I want to work with Python and Kubernetes"
  "looking for a company that uses React"
  "interested in AWS-based projects"
  "would love to work in a Go or Rust environment"

Invalid triggers:
  "I know Python" with no forward-looking intent → skills only, not here
  Vague adjectives without a named technology ("modern stack", "cutting-edge tools")
  Infrastructure the candidate doesn't care about ("any stack is fine")

Algorithm:
  Step 1: Detect technology mentions with explicit future / preference framing.
  Step 2: Exclude mentions that describe only current knowledge with no stated desire.
  Step 3: Keep the user's exact casing ("AWS", not "aws"; "React", not "react").
  Step 4: Deduplicate, preserve order.
  Step 5: If none → [].

──────────────────────
CoT example
──────────────────────
Input: "I know Python and Django well. I'd love to work in a Go or Rust
        environment at my next job, and ideally with Kubernetes."

  "I know Python and Django well" → current knowledge, no future framing
      → skills only, NOT desired_tech_stack
  "I'd love to work in a Go or Rust environment" → explicit future desire → include
  "ideally with Kubernetes" → future preference → include
  → desired_tech_stack = ["Go", "Rust", "Kubernetes"]
  → skills = ["Python", "Django"]   ← populated separately

──────────────────────
Few-shots
──────────────────────
Input: "Looking for a role where I can use React and TypeScript.
        I also have experience with Vue."
Output: {{ "desired_tech_stack": ["React", "TypeScript"] }}
Note: Vue → skills only (experience, no forward intent).

Input: "Interested in companies using AWS or GCP. I currently work with Azure."
Output: {{ "desired_tech_stack": ["AWS", "GCP"] }}

Input: "I want to work with Python — I already know it and want to keep using it."
Output: {{ "desired_tech_stack": ["Python"] }}
Note: Python → both skills AND desired_tech_stack (user explicitly states desire to continue).

----- job_description.preferred_domains -----

List of subject areas, disciplines, or domains the candidate explicitly states
they want to work in. Captures the intellectual and professional territory the
candidate wants to operate in, not the specific tasks or tools.

Distinction from preferred_activities:
  preferred_activities — concrete tasks the candidate wants to DO (verbs)
  preferred_domains    — fields and disciplines the candidate wants to work IN (nouns)

Distinction from desired_tech_stack:
  desired_tech_stack   — specific tools and technologies
  preferred_domains    — broader subject areas that may span many tools

Valid triggers:
  "My main focus is on..."
  "I am interested in..."
  "I want to work in the field of..."
  "passionate about..."
  "specializing in..."
  "background in..." when paired with forward-looking intent

Invalid triggers:
  Specific tools or frameworks ("Python", "PyTorch") → desired_tech_stack
  Concrete tasks ("developing predictive models") → preferred_activities
  Industry or sector ("healthcare", "fintech") → preferred_companies.industry
  Vague adjectives without a named domain ("interesting work", "challenging problems")

Algorithm:
  Step 1: Detect mentions of subject areas, disciplines, or professional domains.
  Step 2: Exclude specific tools (→ desired_tech_stack) and concrete tasks (→ preferred_activities).
  Step 3: Exclude industry/sector mentions (→ preferred_companies.industry).
  Step 4: Keep the user's wording, normalized to a short noun phrase.
  Step 5: Deduplicate, preserve order.
  Step 6: If none → [].

──────────────────────
CoT example
──────────────────────
Input:
  "I am looking for a position as a Data Scientist. My main focus is on applied
   machine learning, statistical analysis, and data-driven product improvements.
   I would like to work in healthcare."

  "applied machine learning" → subject domain, forward-looking focus signal → include
  "statistical analysis" → subject domain, forward-looking focus signal → include
  "data-driven product improvements" → vague, no named discipline → exclude
  "healthcare" → industry/sector → preferred_companies.industry, NOT here
  → preferred_domains = ["applied machine learning", "statistical analysis"]

──────────────────────
Few-shots
──────────────────────
Input: "Passionate about computer vision and robotics."
Output: {{ "preferred_domains": ["computer vision", "robotics"] }}

Input: "I want to work in NLP, specifically on dialogue systems."
Output: {{ "preferred_domains": ["NLP", "dialogue systems"] }}

Input: "Looking for a role using Python and PyTorch in the fintech sector."
Output: {{ "preferred_domains": [] }}
Note: Python, PyTorch → desired_tech_stack; fintech → preferred_companies.industry.

----- job_description.preferred_activities -----

List of activities, tasks, or responsibilities the candidate explicitly states
they want to perform in the new role. Captures desired future work content,
not past experience.

Distinction from candidate_description.skills:
  skills               — tools and technologies the candidate already has
  preferred_activities — actions and tasks the candidate wants to do

Distinction from desired_tech_stack:
  desired_tech_stack   — technologies the candidate wants to work with
  preferred_activities — what the candidate wants to DO (verbs, not tools)

Distinction from preferred_domains:
  preferred_domains    — fields and disciplines the candidate wants to work IN (nouns)
  preferred_activities — concrete tasks the candidate wants to DO (verbs)

Valid triggers:
  "I am particularly interested in roles that involve..."
  "I'd love to work on..."
  "looking for a position where I can..."
  "I want to focus on..."
  "interested in roles that require..."

Invalid triggers:
  Past experience descriptions ("I have been building...", "I worked on...") → skills or responsibilities
  Vague intent without named activity ("I want an interesting job")
  Desired tools without an activity ("I want to work with Python") → desired_tech_stack

Algorithm:
  Step 1: Detect forward-looking activity mentions (desire, interest, intent signals).
  Step 2: Extract the concrete activity as a short action phrase starting with
          a verb or gerund ("developing predictive models", "collaborate with
          product teams"). Strip filler ("roles that involve", "positions where I can").
  Step 3: Exclude mentions that describe only tools without an action.
  Step 4: Exclude past-tense descriptions.
  Step 5: Deduplicate, preserve order.
  Step 6: If none → [].

──────────────────────
CoT example
──────────────────────
Input:
  "I am particularly interested in roles that involve working with large-scale
   datasets, developing predictive models, and collaborating closely with
   engineering and product teams to translate business problems into analytical
   solutions. My main focus is on applied machine learning and statistical analysis."

  "roles that involve working with large-scale datasets"
      → forward-looking, concrete activity → "working with large-scale datasets"
  "developing predictive models"
      → forward-looking, concrete activity → "developing predictive models"
  "collaborating closely with engineering and product teams..."
      → forward-looking, concrete activity → "collaborating with engineering and product teams"
  "My main focus is on applied machine learning and statistical analysis"
      → subject domains, not concrete tasks → preferred_domains, NOT here → exclude

──────────────────────
Few-shots
──────────────────────
Input: "I'd love to work on NLP pipelines and deploy models to production."
Output: {{ "preferred_activities": ["working on NLP pipelines", "deploying models to production"] }}

Input: "Looking for a role where I can lead a data team and define the ML strategy."
Output: {{ "preferred_activities": ["leading a data team", "defining ML strategy"] }}

Input: "I have been building recommendation systems for 3 years."
Output: {{ "preferred_activities": [] }}

----- job_description.preferred_companies -----

A list of CompanyPreference objects. Each object captures a desired industry /
sector / place-type AND/OR a desired location for the job.

Named companies (e.g. "Google", "Tesla") are NOT stored here — they belong
nowhere in this schema; discard them silently.

CompanyPreference fields:
  industry  — preferred industry, sector, or concrete place type
              (e.g. "fintech", "healthcare", "gas station", "pharmacy")
  location  — preferred geographic location for the job
              (city, country, or region)

When to create multiple objects:
  Create one CompanyPreference per distinct (industry, location) pairing the
  user describes. If the user gives two industries with no location constraint,
  create two objects each with location=null.

Algorithm:
  Step 1: Detect all desired industry / sector / place-type mentions.
  Step 2: Detect all desired location mentions.
  Step 3: Pair them if the user links them; otherwise create separate objects.
  Step 4: Discard named companies entirely.
  Step 5: If nothing qualifies → [].

──────────────────────
CoT example
──────────────────────
Input: "I want to work at Google or at a gas station. Preferably in Berlin."
  Named companies → Google → discard
  Place type → "gas station" → industry = "gas station"
  Location → "Berlin" → location = "Berlin"
  User links the gas station preference to Berlin → one object
  → preferred_companies = [{{ "industry": "gas station", "location": "Berlin" }}]

──────────────────────
Few-shots
──────────────────────
Input: "Looking for jobs in fintech or healthcare."
Output: {{
  "preferred_companies": [
    {{ "industry": "fintech", "location": null }},
    {{ "industry": "healthcare", "location": null }}
  ]
}}

Input: "I want a role in pharma, ideally in Zurich."
Output: {{
  "preferred_companies": [
    {{ "industry": "pharma", "location": "Zurich" }}
  ]
}}

Input: "Interested in working at Netflix or Spotify."
Output: {{ "preferred_companies": [] }}


----- job_description.desired_compensation_monthly.salary_min / salary_max -----

Minimum and maximum desired MONTHLY salary as integers. Apply conversion if
the user states a different period.

Conversion:
  annual → monthly : divide by 12, round to nearest integer
  daily  → monthly : multiply by 22 (working days), round

Extraction rules:
  "300,000 rubles" (single value, no range) → salary_min = 300000, salary_max = null
  "100k–120k"      → salary_min = 100000,  salary_max = 120000
  "at least 5000"  → salary_min = 5000,    salary_max = null
  "up to 8000"     → salary_min = null,    salary_max = 8000

Algorithm:
  Step 1: Find all numeric salary mentions.
  Step 2: Detect the period (monthly / annual / daily / per hour).
  Step 3: Convert to monthly if needed.
  Step 4: Assign to min / max based on phrasing.
  Step 5: If no salary mentioned → both null.

──────────────────────
CoT example
──────────────────────
Input: "salary 300 thousand rubles per second"
  Detect numeric → 300,000
  Period → "per second" — treat as literal, not an error
  Conversion: 300,000 RUB/sec × 60 sec × 60 min × 24 hr × 30 days = 777,600,000,000 RUB/month
  → salary_min = 777600000000, salary_max = null, currency = "RUB"

  Note: when a pay period is physically valid but unusual, compute it exactly.
  Do NOT substitute a "more reasonable" period — use what the user stated.

──────────────────────
Few-shots
──────────────────────
Input: "I'm looking for a salary of at least 5,000 EUR per month."
Output: {{ "salary_min": 5000, "salary_max": null }}

Input: "Expecting 100k–120k USD annually."
Output: {{ "salary_min": 8333, "salary_max": 10000 }}

Input: "Anywhere between $4,000 and $6,000 a month."
Output: {{ "salary_min": 4000, "salary_max": 6000 }}


----- job_description.desired_compensation_monthly.currency -----

Three-letter ISO 4217 currency code, extracted ONLY from an explicit mention.
Do NOT infer currency from location, language, or company.

Valid triggers:
  "$", "USD", "dollars"       → "USD"
  "€", "EUR", "euros"         → "EUR"
  "rubles", "₽", "RUB"       → "RUB"
  "£", "GBP", "pounds"        → "GBP"

Invalid triggers:
  Country or city ("Berlin" ≠ EUR)
  Company nationality
  Language of the message

Algorithm:
  Step 1: Find an explicit currency symbol or name.
  Step 2: Map to ISO 4217.
  Step 3: If none found → null.

──────────────────────
Few-shots
──────────────────────
Input: "Salary around 85,000. I'm open to working from anywhere in the EU."
Output: {{ "currency": null }}

Input: "Looking for 300k rubles per month."
Output: {{ "currency": "RUB" }}

Input: "Expecting at least $8,000."
Output: {{ "currency": "USD" }}


----- job_description.desired_compensation_monthly.is_gross -----

True if the user explicitly says the salary is before taxes (gross / до вычета
налогов). False if explicitly after taxes (net / на руки). Null if unspecified.

Valid triggers for True  : "gross", "before tax", "до налогов", "brutto"
Valid triggers for False : "net", "after tax", "на руки", "netto", "take-home"

Algorithm:
  Step 1: Detect explicit gross/net mention.
  Step 2: Map to True / False.
  Step 3: If none → null.

──────────────────────
Few-shots
──────────────────────
Input: "I want 5,000 EUR net."
Output: {{ "is_gross": false }}

Input: "Salary expectation: 120,000 RUB gross."
Output: {{ "is_gross": true }}

Input: "Looking for around 3,000 USD a month."
Output: {{ "is_gross": null }}


----- job_description.desired_compensation_monthly.benefits -----

List of desired non-salary benefits explicitly mentioned. Keep user's wording.

Valid triggers:
  "health insurance", "stock options", "remote work allowance",
  "gym membership", "relocation package", "meal vouchers"

Invalid triggers:
  Remote work policy → goes to preferred_work_mode, not here
  Salary mentions

Algorithm:
  Step 1: Find non-salary perks the user explicitly wants.
  Step 2: Exclude work-mode preferences.
  Step 3: Deduplicate.
  Step 4: If none → [].

──────────────────────
Few-shots
──────────────────────
Input: "I'd love health insurance and stock options."
Output: {{ "benefits": ["health insurance", "stock options"] }}

Input: "Good salary and remote work would be ideal."
Output: {{ "benefits": [] }}


----- job_description.preferred_work_mode.employment_type -----

Map all explicitly stated employment types to a list of EmploymentType enum values:
  full_time, part_time, contract, fixed_term, freelance,
  internship, temporary, self_employed

Default: ["full_time"] — only if WorkModePreference is being created but no
employment type is mentioned.
If no work-mode preference is mentioned at all → preferred_work_mode = null.

Algorithm:
  Step 1: Detect all explicit employment type mentions.
  Step 2: Map each to the closest enum value (table below).
  Step 3: Deduplicate, preserve order.
  Step 4: If WorkModePreference is otherwise populated but type is absent → ["full_time"].
  Step 5: If no work-mode context at all → preferred_work_mode = null.

Mapping table:
  "full-time", "permanent", "full time"  → full_time
  "part-time", "part time"               → part_time
  "contract", "contractor"               → contract
  "fixed-term", "temporary contract"     → fixed_term
  "freelance", "freelancer"              → freelance
  "internship", "intern"                 → internship
  "temporary", "temp"                    → temporary
  "self-employed", "own business"        → self_employed

──────────────────────
CoT example
──────────────────────
Input: "I'm looking for a remote full-time position or freelance projects."
  Detect mentions → "full-time" → full_time, "freelance" → freelance
  Both are explicit → keep both
  → employment_type = ["full_time", "freelance"]

──────────────────────
Few-shots
──────────────────────
Input: "Looking for a full-time remote role."
Output: {{ "employment_type": ["full_time"] }}

Input: "Open to freelance or contract work."
Output: {{ "employment_type": ["freelance", "contract"] }}

Input: "I need a part-time job, preferably hybrid."
Output: {{ "employment_type": ["part_time"] }}


----- job_description.preferred_work_mode.preferred_remote_policy -----
----- job_description.preferred_work_mode.acceptable_remote_policy -----

Both fields are lists drawn from the set: ["remote", "hybrid", "onsite"].

preferred_remote_policy  — what the user explicitly wants most.
acceptable_remote_policy — what the user explicitly states they are willing to
                           accept, excluding any values already in preferred.

Core invariant:
  acceptable_remote_policy must never contain a value that is in preferred_remote_policy.
  If the user mentions a mode only once with no preference signal → it goes into
  acceptable only.

Algorithm:
  Step 1: Detect all remote-policy mentions.
  Step 2: Classify each mention:
      A) Explicit preference signal ("prefer", "love", "want", "ideally") → preferred
      B) Explicit tolerance signal ("open to", "fine with", "could do", "okay with") → acceptable
      C) No signal, single mention → acceptable
  Step 3: Remove from acceptable any value already present in preferred.
  Step 4: If no mentions at all → both lists = [].
  Step 5: Preserve canonical order within each list: remote → hybrid → onsite.

Special cases:
  "only X" / "must be X" / "exclusively X" → preferred = ["X"], acceptable = []
  "doesn't matter" / "any work mode"       → preferred = [], acceptable = ["remote","hybrid","onsite"]

Mapping:
  "work from home", "WFH", "fully remote", "remote-first" → "remote"
  "office 2–3 days", "partly remote", "mixed"             → "hybrid"
  "on-site", "in-office", "no remote"                     → "onsite"

──────────────────────
CoT example
──────────────────────
Input: "I prefer to work fully remotely, but I could do hybrid if needed."
  "fully remotely" + "prefer" signal → preferred = ["remote"]
  "could do hybrid" → tolerance signal → candidate for acceptable
  "hybrid" not in preferred → keep in acceptable
  → preferred_remote_policy  = ["remote"]
  → acceptable_remote_policy = ["hybrid"]

──────────────────────
Few-shots
──────────────────────
Input: "Only interested in remote positions."
Output: {{
  "preferred_remote_policy":  ["remote"],
  "acceptable_remote_policy": []
}}

Input: "I'd love remote, and I'm open to hybrid or onsite too."
Output: {{
  "preferred_remote_policy":  ["remote"],
  "acceptable_remote_policy": ["hybrid", "onsite"]
}}

Input: "Looking for a hybrid or onsite role."
Output: {{
  "preferred_remote_policy":  [],
  "acceptable_remote_policy": ["hybrid", "onsite"]
}}

Input: "Office, remote, doesn't matter to me."
Output: {{
  "preferred_remote_policy":  [],
  "acceptable_remote_policy": ["remote", "hybrid", "onsite"]
}}

"""

═══════════════════════════════
# 4.OUTPUT SHAPE
═══════════════════════════════

In [36]:
output_shape = """

═══════════════════════════════
4. OUTPUT SHAPE
═══════════════════════════════
Always return a single JSON object matching JobAndCandidateDescription.
No extra keys, no markdown, no explanation.

{
  "candidate_description": {
    "years_experience": <int|null>,
    "languages":        <List[str]>,
    "education_level":  <str|null>,
    "skills":           <List[str]>
  },
  "job_description": {
    "desired_positions":   <List[str]>,
    "desired_tech_stack":  <List[str]>,
    "preferred_domains":   <List[str]>,
    "preferred_activities": <List[str]>,
    "preferred_companies": [
      {
        "industry": <str|null>,
        "location": <str|null>
      },
      ...
    ],
    "desired_compensation_monthly": {
      "salary_min": <int|null>,
      "salary_max": <int|null>,
      "currency":   <str|null>,
      "is_gross":   <bool|null>,
      "benefits":   <List[str]>
    } | null,
    "preferred_work_mode": {
      "employment_type":          <List["full_time"|"part_time"|"contract"|"fixed_term"|"freelance"|"internship"|"temporary"|"self_employed"]>,
      "preferred_remote_policy":  <List["remote"|"hybrid"|"onsite"]>,
      "acceptable_remote_policy": <List["remote"|"hybrid"|"onsite"]>
    } | null
  }
}

Constraints:
  - candidate_description and job_description are always present, never null.
  - All List fields default to [] when nothing is extracted.
  - desired_compensation_monthly is null when no salary or benefits are mentioned.
  - preferred_work_mode is null when no employment type or remote policy is mentioned.
  - acceptable_remote_policy never contains a value already in preferred_remote_policy.
  - employment_type values must be from the EmploymentType enum exactly as listed.
  - currency must be a 3-letter ISO 4217 code ("USD", "EUR", "RUB", "GBP", etc.) or null.
  - preferred_companies is [] when no industry or location preference is mentioned.
"""

═══════════════════════════════
# FINAL SYSTEM PROMPT
═══════════════════════════════

In [37]:
system_prompt = f"""

{common_task}
{general_rules}
{candidate_description}
{job_description}
{output_shape}


═══════════════════════════════
JOB REQUEST
═══════════════════════════════
job_request = {user_prompt}
"""

# CLASSES

In [38]:
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field


class EmploymentType(str, Enum):
    FULL_TIME = "full_time"
    PART_TIME = "part_time"
    CONTRACT = "contract"
    FIXED_TERM = "fixed_term"
    FREELANCE = "freelance"
    INTERNSHIP = "internship"
    TEMPORARY = "temporary"
    SELF_EMPLOYED = "self_employed"


class CandidateDescription(BaseModel):
    """
    Factual and relatively stable information about the candidate.
    Used for matching and ranking against job postings.
    """

    years_experience: Optional[int] = Field(
        default=None,
        description="Total years of professional work experience"
    )

    languages: List[str] = Field(
        default_factory=list,
        description="Languages the candidate can speak or work in"
    )

    education_level: Optional[str] = Field(
        default=None,
        description="Highest completed education level (e.g. bachelor's, master's, PhD)"
    )

    skills: List[str] = Field(
        default_factory=list,
        description="Technical and soft skills of the candidate"
    )


class CompanyPreference(BaseModel):
    industry: Optional[str] = Field(
        default=None,
        description="Preferred industry or sector (e.g. fintech, healthcare)"
    )

    location: Optional[str] = Field(
        default=None,
        description="Preferred company or job location"
    )


class CompensationPreference(BaseModel):
    salary_min: Optional[int] = Field(
        default=None,
        description="Minimum desired monthly salary"
    )

    salary_max: Optional[int] = Field(
        default=None,
        description="Maximum desired monthly salary"
    )

    currency: Optional[str] = Field(
        default=None,
        description="Salary currency (e.g. USD, EUR, RUB)"
    )

    is_gross: Optional[bool] = Field(
        default=None,
        description="True if salary is before taxes (gross), False if after taxes (net), None if unspecified"
    )

    benefits: List[str] = Field(
        default_factory=list,
        description="Desired benefits (e.g. health insurance, stock options)"
    )


class WorkModePreference(BaseModel):
    employment_type: List[EmploymentType] = Field(
        description="Type of employment (full_time, contract, freelance, etc.)"
    )

    preferred_remote_policy: List[str] = Field(
        default_factory=list,
        description="Preferred work mode: remote, hybrid, onsite"
    )

    acceptable_remote_policy: List[str] = Field(
        default_factory=list,
        description="List of acceptable work modes (remote, hybrid, onsite)"
    )

class JobDescription(BaseModel):
    """
    What the candidate is looking for in a job.
    This includes preferences, constraints, and search filters.
    """
    desired_positions: List[str] = Field(
        default_factory=list,
        description="Desired job titles, roles, or position names the candidate wants to search for (e.g. Data Scientist, Backend Engineer, Product Manager)"
    )

    # поле desired_tech_stack вероятно будет заполняться редко, но если будет, то улутшит матчинг вакансий
    desired_tech_stack: List[str] = Field(
    default_factory=list,
    description="Preferred technologies, tools, frameworks, or technical environments the candidate wants to work with at the company (e.g. Python, AWS, React, Kubernetes)"
    )

    preferred_domains: List[str] = Field(default_factory=list,
    description=(
        "Subject areas, disciplines, or domains the candidate "
        "wants to work in (e.g. applied machine learning, "
        "computer vision, NLP, statistical analysis)"
        )
    )

    preferred_activities: List[str] = Field(default_factory=list,
        description=(
            "Activities, tasks, or responsibilities the candidate "
            "explicitly wants to do in the new role"
        )
    )

    preferred_companies: List[CompanyPreference] = Field(
        default_factory=list,
        description="Target companies, industries, or locations"
    )

    desired_compensation_monthly: Optional[CompensationPreference] = Field(
        default=None,
        description="Salary and benefits expectations"
    )

    preferred_work_mode: Optional[WorkModePreference] = Field(
        default=None,
        description="Employment type and work arrangement preferences"
    )

class JobAndCandidateDescription(BaseModel):
    candidate_description: CandidateDescription = Field(
        description="Factual and relatively stable information about the candidate"
    )

    job_description: JobDescription = Field(
        description="What the candidate is looking for in a job"
    )

In [39]:
result = parse_mistral("mistral-small-latest", system_prompt, JobAndCandidateDescription)
pretty_print(result)

JobAndCandidateDescription:
  ├── candidate_description:
    ├── years_experience: 9
    ├── languages: ['English']
    ├── education_level: bachelor's
    ├── skills: ['Database Validation', 'Backend Testing', 'Proxyman', 'SQL', 'Kafka', 'Postman', 'test environment administration', 'team management', 'documentation maintenance', 'cross-browser testing', 'release management', 'security testing', 'Windows network configuration']
  ├── job_description:
    ├── desired_positions: ['Senior QA Engineer', 'QA Engineer']
    ├── desired_tech_stack: ['Python']
    ├── preferred_domains: ['FinTech', 'B2B', 'high-load systems', 'backend testing', 'API testing', 'event-driven systems', 'third-party integrations', 'payment systems', 'database validation']
    ├── preferred_activities: ['validating critical business logic', 'improving QA workflows', 'testing backend and API components', 'working with databases', 'testing event-driven systems', 'validating transaction flows', 'testing payment flows

/tmp/ipykernel_1555/1502221593.py:11: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  fields = obj.model_fields


TODO

1 - done. Научить извлекать сведения о зп

2 - done. Вообще не хочет извлекать инфо о компаниях (мб нужен некий пайп LLM → validator → output)

3 - done. в responsibilities какой-то шлак, туда сгружено все, и пожелания к новому месту и активности на старом

4. не уверена насчет сущности preferred_companies.[x], мб хочу выкинуть location и оставить только лист industry-s

5. плохо работает с preferred и acceptable _remote_policy